In [6]:
import pandas as pd
from load_data import load_data
from pathlib import Path

In [19]:
from load_data import load_data
from preprocess import add_bag_of_words_column, STOP_WORDS, LEMMATIZER
from frequency import get_wordcounts
from matching import compute_tfidf_similarity, count_matching_keywords_no_repeats, encoder_scoring


import os
from recommendation import recommend_skills
import pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM

In [ ]:
job_df = load_data('embeddings_Jobs.csv')
job_df

,Unnamed: 0,title,company,announcement,description,embeddings
0,0,"Senior Analyst, Data Science and Analytics",TransUnion,The Muse,TransUnion's Job Applicant Privacy Notice Wha...,[-5.97853363e-02 -5.58718257e-02 -3.90894189e-...
1,1,Senior Data Scientist,"Grubhub Holdings, Inc.",ZipRecruiter,About The Opportunity We're all about connect...,[ 1.29870791e-02 -1.00220166e-01 -1.61358137e-...
2,2,Lead Data Science Analyst,Discover Financial Services,LinkedIn,"Discover. A brighter future. With us, you’ll ...",[-7.51475692e-02 9.18946508e-03 -6.40961602e-...
3,3,Data Science Intern,AbelsonTaylor,Startup Jobs,Are you a 2023 college graduate or rising coll...,[-3.81442308e-02 -3.62408273e-02 1.24055687e-...
4,4,Data Scientist,NORC at the University of Chicago,SimplyHired,"JOB DESCRIPTION: At NORC, Data Scientists pla...",[-9.18436497e-02 1.38543732e-02 -3.48334761e-...
...,...,...,...,...,...,...
785,785,Research and Data Specialist,GovernmentJobs.com,Learn4Good.com,"Description $3,000 hiring bonus to join the J...",[-6.05423264e-02 5.30037135e-02 -5.82620725e-...
786,786,Quality Assurance Data Specialist,Metrocare Services,Glassdoor,Are you looking for a purpose-driven career? A...,[-4.07249182e-02 -4.62232940e-02 3.41084003e-...
787,787,Senior Data Analyst,Gopuff,Startup Jobs,The Senior Data Analyst will join as an analyt...,[-2.07131021e-02 -3.39777879e-02 -7.63482898e-...
788,788,Cost Controller/Data Analyst,Petroplan,Petroplan,Overview: The Cost Controller / Data Analyst p...,[-8.44527502e-03 -1.29908789e-02 -2.20001340e-...


In [ ]:
###
#
# job_df = load_data('../data/job_title_des.csv')

job_df = load_data('../data/Jobs.csv')

resume_df = load_data('../data/gpt_dataset.csv')

#select resume and job description instance from df
test_job_instance = job_df.iloc[9, 2] #change first number to change resume instance (e.g [10,2] for the resume in index 10)

test_resume_instance = resume_df.iloc[34,1] #change first number to change resume instance (e.g [10,1] for the resume in index 10)
# test_resume_instance = resume_df[resume_df['Category']=='ENGINEERING'].iloc[5,1]

#use category=engineering to get resumes with datascience keywords
# test_resume_id = resume_df[resume_df['Resume_str']==test_resume_instance].iloc[0,0]

#use data from gpt_dataset
test_resume_id = resume_df[resume_df['Resume']==test_resume_instance].iloc[0,0]

In [34]:
job_df

,Unnamed: 0,title,company,announcement,description
0,0,"Senior Analyst, Data Science and Analytics",TransUnion,The Muse,TransUnion's Job Applicant Privacy Notice Wha...
1,1,Senior Data Scientist,"Grubhub Holdings, Inc.",ZipRecruiter,About The Opportunity We're all about connect...
2,2,Lead Data Science Analyst,Discover Financial Services,LinkedIn,"Discover. A brighter future. With us, you’ll ..."
3,3,Data Science Intern,AbelsonTaylor,Startup Jobs,Are you a 2023 college graduate or rising coll...
4,4,Data Scientist,NORC at the University of Chicago,SimplyHired,"JOB DESCRIPTION: At NORC, Data Scientists pla..."
...,...,...,...,...,...
785,785,Research and Data Specialist,GovernmentJobs.com,Learn4Good.com,"Description $3,000 hiring bonus to join the J..."
786,786,Quality Assurance Data Specialist,Metrocare Services,Glassdoor,Are you looking for a purpose-driven career? A...
787,787,Senior Data Analyst,Gopuff,Startup Jobs,The Senior Data Analyst will join as an analyt...
788,788,Cost Controller/Data Analyst,Petroplan,Petroplan,Overview: The Cost Controller / Data Analyst p...


In [35]:
test_encoder_scoring(test_job_instance, test_resume_instance)

array([0.10732906], dtype=float32)

In [36]:
def test_tf_idf(test_job=test_job_instance, test_resume=test_resume_instance):
    return compute_tfidf_similarity(test_job, test_resume)

def test_count_matching_keywords_no_repeats(test_job=test_job_instance, test_resume=test_resume_instance):
    return count_matching_keywords_no_repeats(test_job, test_resume)

def test_encoder_scoring(test_job=test_job_instance, test_resume=test_resume_instance, model=None):
    return encoder_scoring(test_job, test_resume, model)

In [37]:
def test_all_scoring_functions(test_resume=test_resume_instance, job_df=job_df, save_csv=True):

    top10_idf = pd.DataFrame()
    top10_encoder = pd.DataFrame()


    #TFIDF
    job_df['tfidf_score'] = job_df['description'].apply(lambda x: test_tf_idf(x, test_resume))
    t = job_df.sort_values(by='tfidf_score', ascending=False).head(50) #higher is better

    top10_idf['idf_job_index'] = t.index
    top10_idf[['idf_job_descriptions', 'tfidf_score']] = t[['description', 'tfidf_score']].values
    #KEYWORD MATCHING
    #job_df['matching_score'] = job_df['Job Description'].apply(lambda x: test_count_matching_keywords_no_repeats(x, test_resume))
    #t = job_df.sort_values(by='matching_score', ascending=False).head(10)

    #top10_df['keyword_matching_job_index'] = t.index
    #top10_df[['keyword_matching_job_descriptions', 'keyword_matching_scores']] = t[['Job Description','matching_score']].values

    print('M: finished keyword matching tests')



    #ENCODER
    model = SentenceTransformer("all-MiniLM-L6-v2")

    ### GET THE EMBEDDINGS
    embeddings = model.encode(job_df['description'][0])

    print("M:finished computing the embeddings")

    job_df['encoder_scores'] = job_df['description'].apply(lambda x: test_encoder_scoring(x, test_resume, model))
    t= job_df.sort_values(by='encoder_scores', ascending=False).head(50)

    top10_encoder['encoder_job_index'] = t.index
    top10_encoder[['encoder_job_descriptions', 'encoder_scores']] = t[['description', 'encoder_scores']].values
    print('M: finished keyword encoder tests')


    return top10_idf, top10_encoder

In [38]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(job_df['description'])
embeddings

array([[-0.05978534, -0.05587183, -0.03908942, ...,  0.03626623,
        -0.07171255,  0.01001317],
       [ 0.01298708, -0.10022017, -0.01613581, ..., -0.05782313,
        -0.11951932, -0.02364505],
       [-0.07514757,  0.00918947, -0.06409616, ..., -0.08773512,
        -0.02639241,  0.00617156],
       ...,
       [-0.0207131 , -0.03397779, -0.07634829, ..., -0.08693486,
         0.06069037,  0.02257267],
       [-0.00844528, -0.01299088, -0.02200013, ..., -0.02987153,
        -0.01147352,  0.03080184],
       [-0.01265353,  0.01352489, -0.02310433, ..., -0.02510767,
        -0.03601936,  0.01253657]], dtype=float32)

In [39]:
def embed_to_column(row):
    row_indx = row.name
    row['embeddings'] = embeddings[row_indx]
    return row

In [40]:
job_df = job_df.apply(embed_to_column, axis = 1)

In [41]:
job_df.to_csv("embeddings_Jobs.csv", index = False)